# SMM4H-HeaRD 2026 Task 2 — Subtask 1: Insomnia Classification

## Qwen3.5-1.7B + LoRA Fine-tuning (T4 GPU, 16GB vRAM)

**Goal:** Classify clinical notes as `yes` (patient has insomnia) or `no`.
**Metric:** F1 score with `yes` as positive class — target > 0.9

**Strategy:**
- Use `Qwen/Qwen3-4B` (instruction-tuned) with 4-bit QLoRA via `unsloth` for efficiency
- Prompt engineering: frame as instruction-following classification
- Chunk long notes (>2000 tokens) and aggregate predictions
- Post-training: if any chunk says `yes`, patient = `yes`
- Generate predictions for val set, test set, and submission JSON

**Fix summary vs v1:**
- Use `Qwen/Qwen3-1.7B` (instruction-tuned) with 4-bit QLoRA via `unsloth` for efficiency
- `MAX_SEQ_LEN` reduced: 2048 → **1024**
- LoRA rank reduced: r=16 → **r=8**
- Batch size: 2 → **1**, grad accum: 4 → **8** (same effective batch=8)
- Set `PYTORCH_ALLOC_CONF=expandable_segments:True` before loading model
- Disabled `packing=True` (packing can spike VRAM unexpectedly on small datasets)
- Text truncation tightened: max_chars 3500 → **1800**
- Added `torch.cuda.empty_cache()` between steps

## Step 1: Set Memory Environment Variables

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment set.")

Environment set.


## Step 2: Install Dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install scikit-learn datasets -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Free VRAM:  {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

GPU: Tesla T4
Total VRAM: 15.64 GB
Free VRAM:  15.53 GB


## Step 3: Upload Data Files

In [ ]:
from google.colab import files
print("Upload: training_corpus.csv, validation_corpus.csv, test_corpus.csv,")
print("        subtask_1_train.json, subtask_1_valid.json")
uploaded = files.upload()

Upload: training_corpus.csv, validation_corpus.csv, test_corpus.csv,
        subtask_1_train.json, subtask_1_valid.json


Saving test_corpus.csv to test_corpus.csv
Saving validation_corpus.csv to validation_corpus.csv
Saving training_corpus.csv to training_corpus.csv
Saving subtask_1_train.json to subtask_1_train.json
Saving subtask_1_valid.json to subtask_1_valid.json


## Step 4: Load and Prepare Data

In [ ]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import f1_score, classification_report

train_df = pd.read_csv("training_corpus.csv")
val_df   = pd.read_csv("validation_corpus.csv")
test_df  = pd.read_csv("test_corpus.csv")

for df in [train_df, val_df, test_df]:
    df["note_id"] = df["note_id"].astype(str)

with open("subtask_1_train.json") as f:
    train_labels = json.load(f)
with open("subtask_1_valid.json") as f:
    val_labels = json.load(f)

train_labeled = train_df[train_df["note_id"].isin(train_labels.keys())].copy()
val_labeled   = val_df[val_df["note_id"].isin(val_labels.keys())].copy()

train_labeled["label"] = train_labeled["note_id"].map(lambda x: train_labels[x]["Insomnia"])
val_labeled["label"]   = val_labeled["note_id"].map(lambda x: val_labels[x]["Insomnia"])

print(f"Train: {len(train_labeled)} | yes={sum(train_labeled.label=='yes')}, no={sum(train_labeled.label=='no')}")
print(f"Val:   {len(val_labeled)}  | yes={sum(val_labeled.label=='yes')}, no={sum(val_labeled.label=='no')}")
print(f"Test:  {len(test_df)}")

Train: 156 | yes=80, no=76
Val:   23  | yes=10, no=13
Test:  1959


## Step 5: Prompt Templates

Truncation budget is now **1800 chars** (≈ 450 tokens) — well within 1024 seq len after adding prompt overhead (~200 tokens).

In [ ]:
SYSTEM_PROMPT = """You are a clinical NLP expert identifying insomnia from clinical notes.

INSOMNIA INDICATORS:
- Direct: insomnia, trouble sleeping, difficulty falling/staying asleep, sleep disturbance, wakes frequently, poor sleep, non-restorative sleep, sleep maintenance
- Medications (strong signal): Zolpidem, Trazodone, Temazepam, Melatonin, Diphenhydramine, Doxepin, Eszopiclone, Ramelteon, Quetiapine (for sleep), Mirtazapine (for sleep), Clonazepam (for sleep)
- Indirect: fatigue, daytime sleepiness, restlessness at night, hyperarousal
- Comorbidities: anxiety, depression, PTSD, chronic pain, restless leg syndrome

RULES:
- If ANY sleep medication is prescribed → likely yes
- If patient explicitly reports sleep problems → yes
- Absence of sleep complaints + no sleep meds → no

Respond ONLY with JSON: {"insomnia": "yes"} or {"insomnia": "no"}"""

MAX_CHARS = 1800  # ← reduced from 3500 to fit in 1024 tokens

def truncate_text(text, max_chars=MAX_CHARS):
    """Keep first 60% + last 40% — preserves drug lists at end and chief complaint at start."""
    if len(text) <= max_chars:
        return text
    head = int(max_chars * 0.6)
    tail = max_chars - head
    return text[:head] + "\n[...]\n" + text[-tail:]

def make_prompt(text, label=None):
    text = truncate_text(text)
    user_msg = f"Clinical note:\n---\n{text}\n---\nInsomnia? Reply only: {{\"insomnia\": \"yes\"}} or {{\"insomnia\": \"no\"}}"
    if label is not None:
        return {"system": SYSTEM_PROMPT, "user": user_msg,
                "assistant": f'{{"insomnia": "{label}"}}'}
    return {"system": SYSTEM_PROMPT, "user": user_msg}

# Verify token budget
sample_text = train_labeled.iloc[0]["text"]
sample_prompt = make_prompt(sample_text, "yes")
full_text = sample_prompt["system"] + sample_prompt["user"] + sample_prompt["assistant"]
# rough token estimate: chars / 4
print(f"Approx tokens for longest sample: {len(full_text)//4} (budget: 1024)")

Approx tokens for longest sample: 688 (budget: 1024)


## Step 6: Load Qwen3-1.7B with Unsloth (4-bit)

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024  # ← reduced from 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

print(f"Free VRAM after load: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Free VRAM after load: 13.99 GB


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,                         # ← reduced from 16
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,               # keep alpha = 2 * r
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.2f}%)")
print(f"Free VRAM after LoRA: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable: 8.7M / 1044M (0.84%)
Free VRAM after LoRA: 13.95 GB


## Step 7: Build HuggingFace Dataset

In [ ]:
from datasets import Dataset

def format_for_training(row):
    prompt = make_prompt(row["text"], row["label"])
    messages = [
        {"role": "system",    "content": prompt["system"]},
        {"role": "user",      "content": prompt["user"]},
        {"role": "assistant", "content": prompt["assistant"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_hf = Dataset.from_pandas(train_labeled[["text", "label"]].reset_index(drop=True))
val_hf   = Dataset.from_pandas(val_labeled[["text", "label"]].reset_index(drop=True))

train_hf = train_hf.map(format_for_training, remove_columns=["text", "label"])
val_hf   = val_hf.map(format_for_training, remove_columns=["text", "label"])

# Verify no sample exceeds our token budget
lengths = [len(tokenizer.encode(x["text"])) for x in train_hf]
print(f"Token lengths — max: {max(lengths)}, mean: {int(np.mean(lengths))}, p95: {int(np.percentile(lengths,95))}")
print(f"Samples exceeding {MAX_SEQ_LEN} tokens: {sum(l > MAX_SEQ_LEN for l in lengths)}")

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Token lengths — max: 1034, mean: 830, p95: 967
Samples exceeding 1024 tokens: 1


## Step 8: Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,             # ← disabled; packing spikes VRAM on small datasets
    args=TrainingArguments(
        per_device_train_batch_size=1,   # ← reduced from 2
        gradient_accumulation_steps=8,   # effective batch = 8
        warmup_ratio=0.1,
        #num_train_epochs=8,              # more epochs to compensate smaller batch
        #learning_rate=2e-4,
        num_train_epochs=12,
        learning_rate=1e-4,               # was 2e-4
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./qwen3_insomnia_lora_v2",
        report_to="none",
        dataloader_pin_memory=False,     # ← saves CPU-GPU transfer overhead
        dataloader_num_workers=0,        # ← avoid fork overhead in Colab
    ),
)

print(f"Free VRAM before training: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/156 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/23 [00:00<?, ? examples/s]

Free VRAM before training: 13.95 GB


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Time:      {trainer_stats.metrics['train_runtime']/60:.1f} min")
print(f"Peak VRAM: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 156 | Num Epochs = 12 | Total steps = 240
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 8,716,288 of 1,729,291,264 (0.50% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,2.787534,2.667470
2,2.033957,1.938102
3,1.669579,1.692954
4,1.448660,1.554195
5,1.370012,1.472131
6,1.261800,1.428051
7,1.169065,1.408374
8,1.170394,1.402068
9,1.127829,1.401375
10,1.117173,1.401928


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Training complete!
Time:      28.4 min
Peak VRAM: 8.08 GB


## Step 9: Inference

In [ ]:
import re

gc.collect()
torch.cuda.empty_cache()
FastLanguageModel.for_inference(model)

def predict_insomnia(text):
    prompt = make_prompt(text)
    messages = [
        {"role": "system", "content": prompt["system"]},
        {"role": "user",   "content": prompt["user"]},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        formatted, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=24,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True).strip()

    try:
        return json.loads(response).get("insomnia", "no").lower()
    except:
        if re.search(r'"insomnia"\s*:\s*"yes"', response, re.IGNORECASE): return "yes"
        if re.search(r'"insomnia"\s*:\s*"no"',  response, re.IGNORECASE): return "no"
        return "yes" if "yes" in response.lower() else "no"

# Quick sanity check
for i in range(3):
    row = train_labeled.iloc[i]
    pred = predict_insomnia(row["text"])
    print(f"Gold={row['label']}, Pred={pred} {'✓' if pred==row['label'] else '✗'}")

Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=24

Gold=yes, Pred=yes ✓


Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gold=no, Pred=no ✓
Gold=no, Pred=no ✓


## Step 10: Evaluate on Validation Set

In [ ]:
from tqdm.auto import tqdm

val_preds = []
val_golds = list(val_labeled["label"])

for _, row in tqdm(val_labeled.iterrows(), total=len(val_labeled), desc="Val inference"):
    val_preds.append(predict_insomnia(row["text"]))

f1 = f1_score(val_golds, val_preds, pos_label="yes")
print(f"\n{'='*50}")
print(f"Validation F1 (yes): {f1:.4f}")
print(f"{'='*50}")
print(classification_report(val_golds, val_preds, target_names=["no", "yes"]))

Val inference:   0%|          | 0/23 [00:00<?, ?it/s]

Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Validation F1 (yes): 0.7273
              precision    recall  f1-score   support

          no       0.82      0.69      0.75        13
         yes       0.67      0.80      0.73        10

    accuracy                           0.74        23
   macro avg       0.74      0.75      0.74        23
weighted avg       0.75      0.74      0.74        23



## Step 11: If F1 < 0.9 — Try Ensemble with Zero-shot Pass

If the fine-tuned model F1 is still below target, this cell runs a **majority vote** between the fine-tuned model and a fresh zero-shot pass (no fine-tuning). With only 23 val examples, a single flip matters a lot.

In [ ]:
# Only run if F1 < 0.9
RUN_PSEUDO_LABELING = True

if RUN_PSEUDO_LABELING:
    import random
    random.seed(42)

    unlabeled = train_df[~train_df["note_id"].isin(train_labels.keys())].copy()
    unlabeled_sample = unlabeled.sample(min(500, len(unlabeled)), random_state=42)

    print(f"Pseudo-labeling {len(unlabeled_sample)} samples...")
    pseudo_preds = []
    for _, row in tqdm(unlabeled_sample.iterrows(), total=len(unlabeled_sample)):
        pseudo_preds.append(predict_insomnia(row["text"]))

    unlabeled_sample["label"] = pseudo_preds

    augmented_train = pd.concat([
        train_labeled[["text", "label"]],
        train_labeled[["text", "label"]],
        unlabeled_sample[["text", "label"]]
    ], ignore_index=True)

    print(f"Augmented train size: {len(augmented_train)}")
    print(augmented_train["label"].value_counts())

    aug_hf = Dataset.from_pandas(augmented_train.reset_index(drop=True))
    aug_hf = aug_hf.map(format_for_training, remove_columns=["text", "label"])

    trainer2 = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=aug_hf,
        eval_dataset=val_hf,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=3,
            learning_rate=5e-5,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=20,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            optim="adamw_8bit",
            output_dir="./qwen3_insomnia_pseudo",
            report_to="none",
        ),
    )
    trainer2.train()

    FastLanguageModel.for_inference(model)
    val_preds2 = [predict_insomnia(row["text"]) for _, row in tqdm(val_labeled.iterrows())]
    f1_2 = f1_score(val_golds, val_preds2, pos_label="yes")
    print(f"\nPost pseudo-label F1: {f1_2:.4f}")
    print(classification_report(val_golds, val_preds2, target_names=["no", "yes"]))

    if f1_2 > f1:
        val_preds = val_preds2
        f1 = f1_2
        print("Using pseudo-labeled model.")
    else:
        print("Original model was better.")

if f1 < 0.9:
    print(f"F1={f1:.4f} < 0.9 — trying temperature ensemble...")

    def predict_with_temp(text, temperature=0.3, n=3):
        prompt = make_prompt(text)
        messages = [
            {"role": "system", "content": prompt["system"]},
            {"role": "user",   "content": prompt["user"]},
        ]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            formatted, return_tensors="pt",
            truncation=True, max_length=MAX_SEQ_LEN
        ).to("cuda")

        votes = []
        for _ in range(n):
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=24,
                    do_sample=True,
                    temperature=temperature,
                    top_p=0.95,
                    pad_token_id=tokenizer.eos_token_id,
                )
            gen  = out[0][inputs["input_ids"].shape[1]:]
            resp = tokenizer.decode(gen, skip_special_tokens=True).strip()
            try:
                votes.append(json.loads(resp).get("insomnia", "no").lower())
            except:
                votes.append("yes" if "yes" in resp.lower() else "no")

        return "yes" if votes.count("yes") >= votes.count("no") else "no"

    val_preds_ens = [
        predict_with_temp(row["text"])
        for _, row in tqdm(val_labeled.iterrows(), total=len(val_labeled), desc="Ensemble")
    ]

    f1_ens = f1_score(val_golds, val_preds_ens, pos_label="yes")
    print(f"Ensemble F1: {f1_ens:.4f}")
    print(classification_report(val_golds, val_preds_ens, target_names=["no", "yes"]))

    if f1_ens > f1:
        print("Using ensemble predictions.")
        val_preds = val_preds_ens
        f1 = f1_ens
    else:
        print("Greedy predictions were better.")
else:
    print(f"F1={f1:.4f} ≥ 0.9 ✓ — no ensemble needed.")

Pseudo-labeling 0 samples...


0it [00:00, ?it/s]

Augmented train size: 312
label
yes    160
no     152
Name: count, dtype: int64


Map:   0%|          | 0/312 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/312 [00:00<?, ? examples/s]

...

Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=24


Post pseudo-label F1: 0.7273
              precision    recall  f1-score   support

          no       0.82      0.69      0.75        13
         yes       0.67      0.80      0.73        10

    accuracy                           0.74        23
   macro avg       0.74      0.75      0.74        23
weighted avg       0.75      0.74      0.74        23

Original model was better.
F1=0.7273 < 0.9 — trying temperature ensemble...


Ensemble:   0%|          | 0/23 [00:00<?, ?it/s]

Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=24

Ensemble F1: 0.7826
              precision    recall  f1-score   support

          no       0.90      0.69      0.78        13
         yes       0.69      0.90      0.78        10

    accuracy                           0.78        23
   macro avg       0.80      0.80      0.78        23
weighted avg       0.81      0.78      0.78        23

Using ensemble predictions.


## Step 12: Save Validation Submission

In [ ]:
val_submission = {
    str(row["note_id"]): {"Insomnia": val_preds[i]}
    for i, (_, row) in enumerate(val_labeled.iterrows())
}
with open("subtask_1_predictions_val.json", "w") as f:
    json.dump(val_submission, f, indent=4)

print(f"Saved val predictions. Final F1: {f1:.4f}")

Saved val predictions. Final F1: 0.7826


## Step 13: Generate Test Predictions

In [ ]:
import os

if os.path.exists("subtask_1_test.json"):
    with open("subtask_1_test.json") as f:
        test_template = json.load(f)
    test_ids = list(test_template.keys())
    print(f"Test template: {len(test_ids)} IDs")
else:
    test_ids = test_df["note_id"].astype(str).tolist()
    print(f"No template — predicting all {len(test_ids)} test rows")

test_to_predict = test_df[test_df["note_id"].isin(test_ids)].copy()
print(f"Predicting {len(test_to_predict)} samples...")

gc.collect()
torch.cuda.empty_cache()

test_submission = {}
for _, row in tqdm(test_to_predict.iterrows(), total=len(test_to_predict), desc="Test inference"):
    test_submission[str(row["note_id"])] = {"Insomnia": predict_insomnia(row["text"])}

with open("subtask_1_predictions_test.json", "w") as f:
    json.dump(test_submission, f, indent=4)

yes_n = sum(1 for v in test_submission.values() if v["Insomnia"] == "yes")
no_n  = len(test_submission) - yes_n
print(f"Test done! yes={yes_n} ({100*yes_n/len(test_submission):.1f}%), no={no_n}")

No template — predicting all 1959 test rows
Predicting 1959 samples...


Test inference:   0%|          | 0/1959 [00:00<?, ?it/s]

Both `max_new_tokens` (=24) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=24

Test done! yes=905 (46.2%), no=1054


## Step 14: Validate & Download

In [ ]:
def validate_submission(path):
    with open(path) as f: data = json.load(f)
    errors = []
    for k, v in data.items():
        if "Insomnia" not in v: errors.append(f"Missing Insomnia key: {k}")
        elif v["Insomnia"] not in ["yes", "no"]: errors.append(f"Bad value: {k}={v}")
    if errors:
        print(f" {len(errors)} errors"); [print(e) for e in errors[:5]]
    else:
        print(f" {path}: {len(data)} entries, all valid.")

validate_submission("subtask_1_predictions_val.json")
validate_submission("subtask_1_predictions_test.json")

 subtask_1_predictions_val.json: 23 entries, all valid.
 subtask_1_predictions_test.json: 1959 entries, all valid.


In [ ]:
from google.colab import files
files.download("subtask_1_predictions_test.json")
files.download("subtask_1_predictions_val.json")
print("Downloaded! Submit subtask_1_predictions_test.json to CodaBench → Subtask 1.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded! Submit subtask_1_predictions_test.json to CodaBench → Subtask 1.


---
## Changes from v1 (OOM fixes)

| Setting | v1 | v2 (this) |
|---|---|---|
| `MAX_SEQ_LEN` | 2048 | **1024** |
| LoRA rank `r` | 16 | **8** |
| `lora_alpha` | 32 | **16** |
| `per_device_train_batch_size` | 2 | **1** |
| `gradient_accumulation_steps` | 4 | **8** (same effective batch) |
| `packing` | True | **False** |
| Text `max_chars` | 3500 | **1800** |
| `PYTORCH_ALLOC_CONF` | not set | **expandable_segments:True** |
| `dataloader_pin_memory` | default | **False** |
| Epochs | 6 | **8** (compensates smaller rank) |

**Expected VRAM usage:** ~11-12 GB peak (safe margin on 14.5 GB available T4)